In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *

In [ ]:
X_gt = np.array(pd.read_csv("./data/circle/position.csv"))
Y_div_gt = np.array(pd.read_csv("./data/circle/velocity_divergence.csv"))
Y_curl_gt = np.array(pd.read_csv("./data/circle/velocity_curl.csv"))
Y_spiral_gt = np.array(pd.read_csv("./data/circle/velocity_spiral.csv"))

color = np.sum(np.sqrt(X_gt**2),axis=1)

plot_2d(X_gt, color)

In [ ]:
from sklearn.neighbors import NearestNeighbors

def jaccard_knn_similarity(X_high, X_low, k=30):
    """
    Computes average Jaccard similarity between k-nearest neighbor sets
    in high-dimensional and low-dimensional spaces.

    Parameters:
        X_high (array-like): Original high-dimensional data (n_samples x n_features)
        X_low (array-like): Embedded low-dimensional data (n_samples x n_components)
        k (int): Number of neighbors to consider (excluding self)

    Returns:
        float: Average Jaccard similarity across all points
    """
    # Fit nearest neighbors
    nn_high = NearestNeighbors(n_neighbors=k+1).fit(X_high)
    nn_low = NearestNeighbors(n_neighbors=k+1).fit(X_low)

    knn_high = nn_high.kneighbors(return_distance=False)[:, 1:]  # skip self
    knn_low = nn_low.kneighbors(return_distance=False)[:, 1:]

    # Compute Jaccard similarity for each point
    jaccard_scores = []
    for i in range(X_high.shape[0]):
        set_high = set(knn_high[i])
        set_low = set(knn_low[i])
        intersection = len(set_high & set_low)
        union = len(set_high | set_low)
        jaccard_scores.append(intersection / union if union > 0 else 0.0)

    return np.mean(jaccard_scores)

In [ ]:
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.manifold import MDS
from sklearn.manifold import trustworthiness
from scipy.spatial.distance import cdist

from scripts.perturbation_distance import PerturbDistanceSolver
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="umap.umap_")

# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_div_gt + np.random.normal(scale=Y_noise_std, size=Y_div_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & UMAP embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    reducer = umap.UMAP(
        n_neighbors=neighbors,
        n_components=2,
        metric="precomputed",
        min_dist=0.6,
        random_state=1
    )
    coords_umap = reducer.fit_transform(dist_mat)

    trust_umap = trustworthiness(X_gt, coords_umap, n_neighbors=neighbors)
    jaccard_umap = jaccard_knn_similarity(X_gt, coords_umap, k=neighbors)

    ax_umap = axes[i]
    sc_umap = ax_umap.scatter(coords_umap[:, 0], coords_umap[:, 1],
                              c=color, s=15, alpha=0.8, cmap='viridis')
    ax_umap.set_title(f"UMAP | alpha={alpha:.3f}\nTrust={trust_umap:.2f}, Jaccard={jaccard_umap:.2f}")
    ax_umap.set_xticks([])
    ax_umap.set_yticks([])
    ax_umap.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plot_2d_quiver(X_gt, Y_div_gt, color, s=30, scale=15, alpha=0.6, cmap="viridis")

In [ ]:
plot_2d_quiver(X[:,:2], Y[:,:2], color, s=30, scale=15, alpha=0.6, cmap="viridis")

In [ ]:
from scripts.TPS import *
tps = ThinPlateSpline(coords_umap)
tps.fit(X, dof_target=10)
velocity = tps.project_velocities(Y)
plot_2d_quiver(coords_umap, velocity, color, s=30, scale=10, alpha=0.6, cmap="viridis")

In [ ]:
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_curl_gt + np.random.normal(scale=Y_noise_std, size=Y_curl_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & UMAP embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    reducer = umap.UMAP(
        n_neighbors=neighbors,
        n_components=2,
        metric="precomputed",
        min_dist=0.6,
        random_state=1
    )
    coords_umap = reducer.fit_transform(dist_mat)

    trust_umap = trustworthiness(X_gt, coords_umap, n_neighbors=neighbors)
    jaccard_umap = jaccard_knn_similarity(X_gt, coords_umap, k=neighbors)

    ax_umap = axes[i]
    sc_umap = ax_umap.scatter(coords_umap[:, 0], coords_umap[:, 1],
                              c=color, s=15, alpha=0.8, cmap='viridis')
    ax_umap.set_title(f"UMAP | alpha={alpha:.3f}\nTrust={trust_umap:.2f}, Jaccard={jaccard_umap:.2f}")
    ax_umap.set_xticks([])
    ax_umap.set_yticks([])
    ax_umap.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
plot_2d_quiver(X[:,:2], Y[:,:2], color, s=30, scale=15, alpha=0.6, cmap="viridis")

tps = ThinPlateSpline(coords_umap)
tps.fit(X, dof_target=10)
velocity = tps.project_velocities(Y)
plot_2d_quiver(coords_umap, velocity, color, s=30, scale=10, alpha=0.6, cmap="viridis")

In [ ]:
# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 2  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_spiral_gt + np.random.normal(scale=Y_noise_std, size=Y_spiral_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & UMAP embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    reducer = umap.UMAP(
        n_neighbors=neighbors,
        n_components=2,
        metric="precomputed",
        min_dist=0.6,
        random_state=1
    )
    coords_umap = reducer.fit_transform(dist_mat)

    trust_umap = trustworthiness(X_gt, coords_umap, n_neighbors=neighbors)
    jaccard_umap = jaccard_knn_similarity(X_gt, coords_umap, k=neighbors)

    ax_umap = axes[i]
    sc_umap = ax_umap.scatter(coords_umap[:, 0], coords_umap[:, 1],
                              c=color, s=15, alpha=0.8, cmap='viridis')
    ax_umap.set_title(f"UMAP | alpha={alpha:.3f}\nTrust={trust_umap:.2f}, Jaccard={jaccard_umap:.2f}")
    ax_umap.set_xticks([])
    ax_umap.set_yticks([])
    ax_umap.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
tps.lambda_reg, tps

In [ ]:
plot_2d_quiver(X[:,:2], Y[:,:2], color, s=30, scale=15, alpha=0.6, cmap="viridis")

tps = ThinPlateSpline(coords_umap)
tps.fit(X, dof_target=10)
velocity = tps.project_velocities(Y)
plot_2d_quiver(coords_umap, velocity, color, s=30, scale=10, alpha=0.6, cmap="viridis")

In [ ]:
# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 5  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_div_gt + np.random.normal(scale=Y_noise_std, size=Y_div_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & MDS embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=1)
    coords_mds = mds.fit_transform(dist_mat)

    trust_mds = trustworthiness(X_gt, coords_mds, n_neighbors=neighbors)
    jaccard_mds = jaccard_knn_similarity(X_gt, coords_mds, k=neighbors)

    ax_mds = axes[i]
    sc_mds = ax_mds.scatter(coords_mds[:, 0], coords_mds[:, 1],
                            c=color, s=15, alpha=0.8, cmap='viridis')
    ax_mds.set_title(f"MDS | alpha={alpha:.3f}\nTrust={trust_mds:.2f}, Jaccard={jaccard_mds:.2f}")
    ax_mds.set_xticks([])
    ax_mds.set_yticks([])
    ax_mds.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 5  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_curl_gt + np.random.normal(scale=Y_noise_std, size=Y_curl_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & MDS embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=1)
    coords_mds = mds.fit_transform(dist_mat)

    trust_mds = trustworthiness(X_gt, coords_mds, n_neighbors=neighbors)
    jaccard_mds = jaccard_knn_similarity(X_gt, coords_mds, k=neighbors)

    ax_mds = axes[i]
    sc_mds = ax_mds.scatter(coords_mds[:, 0], coords_mds[:, 1],
                            c=color, s=15, alpha=0.8, cmap='viridis')
    ax_mds.set_title(f"MDS | alpha={alpha:.3f}\nTrust={trust_mds:.2f}, Jaccard={jaccard_mds:.2f}")
    ax_mds.set_xticks([])
    ax_mds.set_yticks([])
    ax_mds.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------------------
X_noise_std = 0.1
Y_noise_std = 0.1
n_dummy_dims = 5  # number of extra dimensions

# ------------------------------------------------------------------------------
# Add noise and dummy dimensions
# ------------------------------------------------------------------------------
X = X_gt + np.random.normal(scale=X_noise_std, size=X_gt.shape)
Y = Y_spiral_gt + np.random.normal(scale=Y_noise_std, size=Y_spiral_gt.shape)

# Add dummy dimensions with Gaussian noise
X_dummy = np.random.normal(scale=X_noise_std, size=(X.shape[0], n_dummy_dims))
Y_dummy = np.random.normal(scale=Y_noise_std, size=(Y.shape[0], n_dummy_dims))

X = np.hstack([X, X_dummy])
Y = np.hstack([Y, Y_dummy])

# Standardize to mean 0, std 1
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = (Y - Y.mean(axis=0)) / Y.std(axis=0)

# ------------------------------------------------------------------------------
# Solver & MDS embeddings
# ------------------------------------------------------------------------------
solver = PerturbDistanceSolver(X, Y)
t_dist = solver.pairwise_time_constrained_L2_distance()
neighbors = 20

alphas = np.logspace(0, 2.5, 3)
alphas = np.concatenate(([0], alphas))
dist_mat_sq = cdist(X, X, metric='sqeuclidean')

fig, axes = plt.subplots(nrows=1, ncols=len(alphas), figsize=(4 * len(alphas), 4))

# Ensure axes is always iterable
if len(alphas) == 1:
    axes = [axes]

for i, alpha in enumerate(alphas):
    dist_mat = np.sqrt(dist_mat_sq + alpha * t_dist ** 2)

    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=1)
    coords_mds = mds.fit_transform(dist_mat)

    trust_mds = trustworthiness(X_gt, coords_mds, n_neighbors=neighbors)
    jaccard_mds = jaccard_knn_similarity(X_gt, coords_mds, k=neighbors)

    ax_mds = axes[i]
    sc_mds = ax_mds.scatter(coords_mds[:, 0], coords_mds[:, 1],
                            c=color, s=15, alpha=0.8, cmap='viridis')
    ax_mds.set_title(f"MDS | alpha={alpha:.3f}\nTrust={trust_mds:.2f}, Jaccard={jaccard_mds:.2f}")
    ax_mds.set_xticks([])
    ax_mds.set_yticks([])
    ax_mds.grid(True)

plt.tight_layout()
plt.show()